In [2]:
import pandas as pd
import numpy as np
import os
import json

# 读取当前目录下的sample.tsv 
df = pd.read_csv('sample.tsv', sep='\t')
df['Time'] = df['Time'].fillna('')
df['Condition'] = df['Strain'] + '_' + df['Carbon Sources'] + '_' + df['Temperature'].fillna('NA') + '_' + df['Time'].fillna('NA')
df.head()

,Sample ID,Sample Name,Experiment,Run,Series Accession,Study Title,Species,Model,Library Seletion,Library Source,Library Strategy,Samples,Strain,Carbon Sources,Temperature,Time,Year,Condition
0,mt001001,GSM675519,SRX043463,SRR101401,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,Illumina Genome Analyzer II,cDNA,transcriptomic,RNA-Seq,Barley_34C_rep1,wt,Barley,34°,,2011 Oct,wt_Barley_34°_
1,mt001002,"GSM675520,GSM675523","SRX043464, SRX043467","SRR101402,SRR101405",GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,Illumina Genome Analyzer II,cDNA,transcriptomic,RNA-Seq,"Barley_45C_rep1,Barley_45C_rep2",wt,Barley,45°,,2011 Oct,wt_Barley_45°_
2,mt001003,"GSM675524,GSM675525","SRX043468, SRX043469","SRR101406,SRR101407",GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,Illumina Genome Analyzer II,cDNA,transcriptomic,RNA-Seq,"Glucose_34C_rep1,Glucose_45C_rep2",wt,Glucose,34°,,2011 Oct,wt_Glucose_34°_
3,mt001004,GSM675521,SRX043465,SRR101403,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,Illumina Genome Analyzer II,cDNA,transcriptomic,RNA-Seq,Glucose_45C_rep1,wt,Glucose,45°,,2011 Oct,wt_Glucose_45°_
4,mt001005,GSM675522,SRX043466,SRR101404,GSE27323,Transcriptomic response to substrate and tempe...,Myceliphothora thermophila,Illumina Genome Analyzer II,cDNA,transcriptomic,RNA-Seq,Alfalfa_45C_rep1,wt,Alfalfa,45°,,2011 Oct,wt_Alfalfa_45°_


In [3]:
# df保留Sample Number、Sample ID、Series ID、Study Title、Species、Strain、Sample Name、Canbon Sources、Temperature列
df_1 = df[['Sample ID', 'Series Accession','Study Title', 'Condition']]

# 如果Conditon最后一位是_，则去掉
df_1.loc[:,'Condition'] = df_1['Condition'].apply(lambda x: x[:-1] if x[-1] == '_' else x)

df_1

,Sample ID,Series Accession,Study Title,Condition
0,mt001001,GSE27323,Transcriptomic response to substrate and tempe...,wt_Barley_34°
1,mt001002,GSE27323,Transcriptomic response to substrate and tempe...,wt_Barley_45°
2,mt001003,GSE27323,Transcriptomic response to substrate and tempe...,wt_Glucose_34°
3,mt001004,GSE27323,Transcriptomic response to substrate and tempe...,wt_Glucose_45°
4,mt001005,GSE27323,Transcriptomic response to substrate and tempe...,wt_Alfalfa_45°
...,...,...,...,...
103,mt014008,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtLat-1_Arabinan_45°_4d
104,mt014009,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra_Aarabinose_45°_4h
105,mt014010,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra_Xylose_45°_4h
106,mt014011,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra_Arabinan_45°_4h


In [4]:
# 读取dataset.tsv
df_2 = pd.read_csv('dataset.tsv', sep='\t')
df_2 = df_2[['Series Accession', 'PMID']]

df_2.head(20)

,Series Accession,PMID
0,GSE213575,37013645.0
1,GSE213573,37013645.0
2,GSE213570,37013645.0
3,GSE221464,36966330.0
4,GSE214002,36691040.0
5,GSE214142,36478865.0
6,GSE205182,36165620.0
7,GSE184074,35257374.0
8,GSE183387,NaN
9,GSE114057,31078793.0


In [5]:
# 将df_1和df_2合并，根据Series ID列
df_3 = pd.merge(df_1, df_2, on='Series Accession', how='left')

# 重置索引
df_3 = df_3.reset_index(drop=True)

df_3['PMID'] = df_3['PMID'].astype(str)
df_3 = df_3[df_3['PMID'] != 'nan']

# PMID去掉最后两个字符
df_3['PMID'] = df_3['PMID'].apply(lambda x: x[:-2] if x[-2:] == '.0' else x)

# 保存df_3为csv文件
df_3
# df_3.to_csv('sample_condition.csv', index=False)

,Sample ID,Series Accession,Study Title,Condition,PMID
0,mt001001,GSE27323,Transcriptomic response to substrate and tempe...,wt_Barley_34°,21964414
1,mt001002,GSE27323,Transcriptomic response to substrate and tempe...,wt_Barley_45°,21964414
2,mt001003,GSE27323,Transcriptomic response to substrate and tempe...,wt_Glucose_34°,21964414
3,mt001004,GSE27323,Transcriptomic response to substrate and tempe...,wt_Glucose_45°,21964414
4,mt001005,GSE27323,Transcriptomic response to substrate and tempe...,wt_Alfalfa_45°,21964414
...,...,...,...,...,...
103,mt014008,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtLat-1_Arabinan_45°_4d,36966330
104,mt014009,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra_Aarabinose_45°_4h,36966330
105,mt014010,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra_Xylose_45°_4h,36966330
106,mt014011,GSE221464,The arabinose transporter MtLat-1 is involved ...,MtAra_Arabinan_45°_4h,36966330


In [8]:
def get_gene_exp(gene):
    """
    根据基因id，返回该基因在各个文献对应样本的TPM表达量
    """

    # 读取exp.tsv和sample_condition.csv，分别为基因的表达量(重复样本取平均值)和基因对应的文献与样本信息
    df_exp_fname = pd.read_csv('exp.tsv', sep='\t')
    df_condition = pd.read_csv('sample_condition.csv', sep=',')

    # 根据gene id从df_exp_fname中取出对应的行，并且将TPM标准化为log2(x + 1)
    df = df_exp_fname[df_exp_fname['Gene id'] == gene].copy()
    # df.iloc[:, 1:] = df.iloc[:, 1:].applymap(lambda x: np.log2(x + 1)).round(2)

    df_condition['Sample Exp'] = df_condition['Sample ID'].map(df.set_index('Gene id').iloc[:, 0:].T.to_dict()[gene])  # 原先是Sample Number，改叫为Sample ID

    # df_condition返回的是一个dataframe；构造字典
    output_dict = {
        'gene_id': {
            'gene_id': gene
        }
    }
    for _, row in df_condition.iterrows():
        pmid = str(row['PMID'])   # 原先是Publications，改叫为PMID
        if pmid not in output_dict['gene_id']:
            output_dict['gene_id'][pmid] = {
                'PMID': pmid,
                'Study Title': row['Study Title'],
            }
        sample_number = row['Sample ID']
        output_dict['gene_id'][pmid][sample_number] = {
            'Sample ID': sample_number,
            'Condition': row['Condition'],
            'Exp': str(row['Sample Exp'])
        }

    # # output_dict转为json格式
    # output_json = json.dumps(output_dict, indent=4)
    
    return output_dict

dict_gene_exp = get_gene_exp('MYCTH_2114025')
dict_gene_exp

{'gene_id': {'gene_id': 'MYCTH_2114025',
  '21964414': {'PMID': '21964414',
   'Study Title': 'Transcriptomic response to substrate and temperature in two thermophilic fungi, Myceliophthora thermophila and Thielavia terrestris, and a related mesophile, Chaetomium globosum',
   'mt001001': {'Sample ID': 'mt001001',
    'Condition': 'wt_Barley_34°',
    'Exp': '2.67'},
   'mt001002': {'Sample ID': 'mt001002',
    'Condition': 'wt_Barley_45°',
    'Exp': '3.55'},
   'mt001003': {'Sample ID': 'mt001003',
    'Condition': 'wt_Glucose_34°',
    'Exp': '2.35'},
   'mt001004': {'Sample ID': 'mt001004',
    'Condition': 'wt_Glucose_45°',
    'Exp': '2.29'},
   'mt001005': {'Sample ID': 'mt001005',
    'Condition': 'wt_Alfalfa_45°',
    'Exp': '4.34'}},
  '24881579': {'PMID': '24881579',
   'Study Title': 'Transcriptomic response to different biomass in Myceliophthora thermophila (Sporotrichum thermophile)',
   'mt002001': {'Sample ID': 'mt002001',
    'Condition': 'wt_Oat_45°',
    'Exp': '3.9'

In [21]:
#将output_dict 转为json文件
with open('gene_exp.json', 'w') as f:
    json.dump(dict_gene_exp, f, indent=4)